# Phase 2: Data Cleaning & Preprocessing Pipeline

**Project:** Indian E-Commerce Recommendation System  
**Objective:** Transform 9 raw, heterogeneous CSV files into a unified, cleaned, deduplicated product catalog localized for Indian E-Commerce.

### Pipeline Steps:
1. **Encoding & Ingestion**: Read raw CSVs with `latin1` encoding to preserve symbols.
2. **Schema & Column Normalization**: Map mismatched column titles (`Product Description `, `Production Description`, `PriceDollar)`).
3. **Artifact Removal**: Drop stray unescaped columns (`Unnamed: 7` through `Unnamed: 15`).
4. **Price Parsing & INR Conversion**: Clean currency strings, discard invalid records (no zero replacement), and convert USD to ₹ INR.
5. **Review Parsing**: Convert review counts into integers, setting missing reviews to 0.
6. **Deduplication**: Remove identical product listings within categories.
7. **Identifier Validation**: Assign deterministic IDs (`IN-PROD-000001` to `IN-PROD-076327`).

In [ ]:
import os
import sys
import pandas as pd
import json

# Add project root to sys.path
sys.path.append(os.path.abspath(".."))
from src.data.clean_dataset import clean_entire_dataset, DEFAULT_USD_TO_INR_RATE

RAW_DIR = os.path.join("..", "data", "raw")
OUTPUT_CSV = os.path.join("..", "data", "processed", "products_cleaned.csv")
SUMMARY_JSON = os.path.join("..", "reports", "cleaning_summary.json")

print(f"USD to INR Conversion Rate: ₹{DEFAULT_USD_TO_INR_RATE}")

## 1. Execute the Pipeline and Load Telemetry

In [ ]:
df_cleaned = clean_entire_dataset(
    raw_dir=RAW_DIR,
    output_csv_path=OUTPUT_CSV,
    summary_report_path=SUMMARY_JSON
)
print(f"Final clean catalog shape: {df_cleaned.shape}")
df_cleaned.head()

## 2. Review Cleaning Summary & Funnel

In [ ]:
with open(SUMMARY_JSON, "r", encoding="utf-8") as f:
    summary = json.load(f)

print(f"Raw total records:              {summary['raw_total_records']:,}")
print(f"After missing price filter:     {summary['post_clean_before_dedup']:,}")
print(f"Duplicates removed:             {summary['duplicates_removed']:,}")
print(f"Final unique products:          {summary['final_cleaned_records']:,}")

# Category distribution
cat_series = pd.Series(summary["categories"])
print("\n--- Products per Category ---")
print(cat_series)

## 3. Data Integrity & Validation Checks
Verify no nulls exist in critical fields (`product_id`, `title`, `category`, `price_inr`).

In [ ]:
critical_fields = ["product_id", "title", "category", "price_inr", "review_count"]
null_counts = df_cleaned[critical_fields].isnull().sum()
print("Null counts in critical fields:")
print(null_counts)

assert null_counts.sum() == 0, "Critical fields contain nulls!"
assert (df_cleaned["price_inr"] <= 0).sum() == 0, "Found non-positive prices!"
assert df_cleaned["product_id"].nunique() == len(df_cleaned), "Product IDs are not unique!"
print("\nAll automated data integrity assertions PASSED.")